# League of Legends: Feature Engineering

Owner: Isaac. Builds on notebook 01 (cleaning, split, first engineered features) and notebook 02 (logistic regression and decision trees).

What notebook 02 showed:
- Objective features (turrets, dragons, kills) added a small but consistent gain beyond gold and experience.
- The 10-to-14 change features added nothing at minute 14.
- Gold, experience, level, and kill differences are highly correlated.

This notebook:
1. Engineers **objective/combat summaries** and **trajectory features** (how the lead developed since minute 10) for **any prediction minute**.
2. Handles the heavy-tailed `wardsPlaced` field instead of dropping it.
3. Screens each new feature, then measures what each feature group adds through an **ablation study** for logistic regression and decision trees.
4. Tries **transformations**: PCA on the collinear resource features, and K-means cluster distances as features.
5. Runs **L1 feature selection**.
6. Repeats the ablation at minutes 10, 14, 18, and 20, and saves feature tables for the random-forest notebook.

The feature code lives in `src/features.py` so other notebooks can reuse it. Only training matches are used; the test matches stay untouched.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Support notebooks launched from the repository root or notebooks/.
root = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(root))

from src.evaluation import METRIC_COLS, SEED, GoldLeadRule, cv_scores, summarize, tune
from src.features import (
    FEATURE_DEFINITIONS, NOTEBOOK01_FEATURES, OBJECTIVE_FEATURES, TRAJECTORY_FEATURES,
    build_features,
)

processed_dir = root / "data" / "processed"
report_dir = root / "reports"
figure_dir = report_dir / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

split_path = processed_dir / "split_ids.csv"
if not split_path.is_file():
    raise FileNotFoundError(f"Missing {split_path}. Run 01_data_exploration.ipynb first.")

raw = pd.read_csv(root / "data" / "raw" / "lol_ranked_games.csv")
split_ids = pd.read_csv(split_path)
train_ids = set(split_ids.loc[split_ids["split"].eq("train"), "gameId"])
print(f"Training matches: {len(train_ids):,}")

## 1. New features

All features for minute *t* use only that match's snapshots at minutes 10, 12, ..., *t*, and only matches still in progress at *t* are kept. Trajectory features need at least three snapshots, so they exist for minute 14 and later.

In [ ]:
definitions = pd.DataFrame(
    [(name, "objective" if name in OBJECTIVE_FEATURES else "trajectory", text)
     for name, text in FEATURE_DEFINITIONS.items()],
    columns=["feature", "group", "definition"],
)
definitions.to_csv(report_dir / "feature_definitions.csv", index=False)
with pd.option_context("display.max_colwidth", None):
    display(definitions)

In [ ]:
table14, groups14 = build_features(raw, 14)
train14 = table14.loc[table14["gameId"].isin(train_ids)].reset_index(drop=True)
print("Minute-14 table:", table14.shape, "| training matches:", len(train14))
print({name: len(cols) for name, cols in groups14.items()})

# Consistency check: the shared definitions reproduce notebook 01's features
nb01 = pd.read_csv(processed_dir / "engineered_data.csv").set_index("gameId")
mine = table14.set_index("gameId").loc[nb01.index]
for new, old in [("goldChange4", "goldChange10to14"), ("expChange4", "expChange10to14"),
                 ("dragonDiff", "dragonDiff"), ("killDeathDiff", "killDeathDiff"),
                 ("outerTurretDiff", "outerTurretDiff")]:
    assert mine[new].equals(nb01[old]), new
print("Matches notebook 01's engineered features.")

### 1.1 `wardsPlaced`: a heavy-tailed field

Notebook 01 excluded `wardsPlaced` after finding matches with 300-400 wards by minute 14, while the median is about 24. Instead of dropping it, we use `log(1 + wardsPlaced)`, which keeps the ordering but shrinks the extreme values. The transform has no fitted parameters, so it cannot leak information.

In [ ]:
wards = raw.loc[raw["frame"].eq(14) & raw["gameId"].isin(train_ids), "wardsPlaced"]
print(wards.describe(percentiles=[0.5, 0.9, 0.99]).round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(wards, bins=80, color="#d97757")
axes[0].set_title("wardsPlaced at minute 14")
axes[0].set_xlabel("Wards placed by blue")
axes[1].hist(np.log1p(wards), bins=80, color="#2a9d8f")
axes[1].set_title("log(1 + wardsPlaced)")
axes[1].set_xlabel("Log wards placed")
for ax in axes:
    ax.set_ylabel("Training matches")
fig.tight_layout()
fig.savefig(figure_dir / "wards_placed_transform.png", dpi=200, bbox_inches="tight")
plt.show()

### 1.2 Screening: how predictive is each feature on its own?

For each engineered feature we compute the ROC AUC of the raw feature value against `hasWon`. An AUC of 0.5 means no signal. Values below 0.5 mean "higher is worse for blue", so we plot the *strength* max(AUC, 1 - AUC) and record the direction. No model is fit here.

In [ ]:
table20, groups20 = build_features(raw, 20)
train20 = table20.loc[table20["gameId"].isin(train_ids)].reset_index(drop=True)

engineered = OBJECTIVE_FEATURES + TRAJECTORY_FEATURES
screen = []
for minute, data in [(14, train14), (20, train20)]:
    for feature in engineered + ["goldDiff"]:
        auc = roc_auc_score(data["hasWon"], data[feature])
        screen.append({"minute": minute, "feature": feature, "strength": max(auc, 1 - auc),
                       "direction": "+" if auc >= 0.5 else "-"})
screen = pd.DataFrame(screen)

order = screen.loc[screen["minute"].eq(14)].sort_values("strength", ascending=False)["feature"]
fig, ax = plt.subplots(figsize=(9, 9))
sns.barplot(data=screen, y="feature", x="strength", hue="minute", order=order,
            palette=["#2a9d8f", "#d97757"], ax=ax)
ax.axvline(0.5, color="gray", linestyle="--", linewidth=1)
ax.set_xlim(0.45, 0.9)
ax.set_xlabel("Single-feature ROC AUC strength, max(AUC, 1 - AUC)")
ax.set_ylabel("")
ax.set_title("Predictive strength of each feature alone (training matches)")
fig.tight_layout()
fig.savefig(figure_dir / "feature_screening_auc.png", dpi=200, bbox_inches="tight")
plt.show()

display(screen.pivot(index="feature", columns="minute", values="strength")
        .loc[order].round(3).join(screen.loc[screen["minute"].eq(14)].set_index("feature")["direction"]))

### 1.3 Does the path to a lead matter?

Two teams can hold the same minute-14 gold lead while getting there differently. Here we split teams that are **ahead** at minute 14 by whether they were **behind at some earlier snapshot** (`minGoldLead < 0`).

In [ ]:
ahead = train14.loc[train14["goldDiff"] > 0].copy()
ahead["lead"] = pd.cut(ahead["goldDiff"], [0, 1000, 2000, 3000, 5000, np.inf],
                       labels=["0-1k", "1-2k", "2-3k", "3-5k", "5k+"])
ahead["history"] = np.where(ahead["minGoldLead"] < 0, "Was behind earlier", "Never behind")
path_table = ahead.groupby(["lead", "history"], observed=True)["hasWon"].agg(matches="size", win_rate="mean")
display(path_table.unstack().round(3))

plot_table = path_table.reset_index()
plot_table = plot_table[plot_table["matches"] >= 30]
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=plot_table, x="lead", y="win_rate", hue="history",
            palette=["#2a9d8f", "#d97757"], ax=ax)
ax.set_xlabel("Blue gold lead at minute 14 (groups with < 30 matches hidden)")
ax.set_ylabel("Blue win rate")
ax.set_title("Same lead, different path: win rate by earlier deficit")
fig.tight_layout()
fig.savefig(figure_dir / "lead_path_win_rates.png", dpi=200, bbox_inches="tight")
plt.show()

## 2. Ablation at minute 14

Each feature set adds one group on top of the previous baseline, so differences show what that group contributes:

| Set | Contents |
| --- | --- |
| Notebook 01 features | raw fields + notebook 01's five engineered features (same as notebook 02's "All features") |
| + objective | + all objective/combat/vision summaries |
| + trajectory | notebook 01 features + all trajectory features |
| All engineered | raw fields + objective + trajectory |
| Compact engineered | gold/experience/level differences + objective + trajectory (no raw counters) |

Same evaluation protocol as notebook 02: 5-fold CV grouped by `gameId`, best hyperparameters chosen by log loss.

In [ ]:
RESOURCES = ["goldDiff", "expDiff", "champLevelDiff"]


def unique(cols):
    return list(dict.fromkeys(cols))


def feature_sets_for(groups, minute):
    raw_cols = groups["raw"]
    nb01_cols = [c for c in NOTEBOOK01_FEATURES if minute >= 14 or c not in TRAJECTORY_FEATURES]
    new = unique(OBJECTIVE_FEATURES + groups["trajectory"])
    candidates = {
        "Notebook 01 features": unique(raw_cols + nb01_cols),
        "+ objective": unique(raw_cols + nb01_cols + OBJECTIVE_FEATURES),
        "+ trajectory": unique(raw_cols + nb01_cols + groups["trajectory"]),
        "All engineered": unique(raw_cols + new),
        "Compact engineered": unique(RESOURCES + new),
    }
    # Drop sets identical to an earlier one (e.g. no trajectory features at minute 10)
    sets, seen = {}, []
    for name, cols in candidates.items():
        if set(cols) not in seen:
            sets[name] = cols
            seen.append(set(cols))
    return sets


lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
lr_grid = {"logisticregression__C": [0.001, 0.01, 0.1, 1, 10]}
dt = DecisionTreeClassifier(random_state=SEED)
# Notebook 02's best trees sat at the edge of its grid (min_samples_leaf=200),
# so this grid extends to larger leaves and deeper trees.
dt_grid = {"max_depth": [4, 6, 8, 10, 12], "min_samples_leaf": [100, 200, 400, 800, 1600]}


def run_ablation(data, groups, minute):
    rows = [summarize("Gold-lead rule", "sign of goldDiff",
                      cv_scores(GoldLeadRule(), data, ["goldDiff"]), minute=minute, n_features=1)]
    for set_name, cols in feature_sets_for(groups, minute).items():
        for model_name, model, grid in [("Logistic regression", lr, lr_grid),
                                        ("Decision tree", dt, dt_grid)]:
            search, scores = tune(model, grid, data, cols)
            rows.append(summarize(model_name, set_name, scores, search.best_params_,
                                  minute=minute, n_features=len(cols)))
    return pd.DataFrame(rows)


ablation14 = run_ablation(train14, groups14, 14)
display(ablation14[["model", "features", "n_features"] + METRIC_COLS + ["params"]].round(4))

In [ ]:
plot_data = ablation14.loc[ablation14["model"].ne("Gold-lead rule")]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, metric, title in [(axes[0], "accuracy", "CV accuracy (higher is better)"),
                          (axes[1], "log_loss", "CV log loss (lower is better)")]:
    sns.barplot(data=plot_data, y="features", x=metric, hue="model",
                palette=["#2a9d8f", "#d97757"], ax=ax)
    lo, hi = plot_data[metric].min(), plot_data[metric].max()
    pad = (hi - lo) * 0.6
    ax.set_xlim(lo - pad, hi + pad)
    ax.set_title(title)
    ax.set_ylabel("")
handles, labels = axes[1].get_legend_handles_labels()
for ax in axes:
    ax.get_legend().remove()
fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False)
fig.suptitle("Feature-set ablation at minute 14 (axes zoomed in to show small differences)")
fig.tight_layout(rect=[0, 0.07, 1, 1])
fig.savefig(figure_dir / "fe_ablation_minute14.png", dpi=200, bbox_inches="tight")
plt.show()

## 3. Transformations for logistic regression (minute 14, all engineered features)

- **PCA on resources:** `goldDiff`, `expDiff`, and `champLevelDiff` are 0.88-0.95 correlated. We replace them with their first one or two principal components, fit inside each training fold.
- **K-means distances:** following notebook 01's clustering of early-game states, we add each match's distance to *k* cluster centers as features. K-means is fit inside each training fold, so the test fold never influences the clusters.

In [ ]:
all14 = feature_sets_for(groups14, 14)["All engineered"]
others = [c for c in all14 if c not in RESOURCES]

pca_lr = make_pipeline(
    ColumnTransformer([
        ("resources", make_pipeline(StandardScaler(), PCA(random_state=SEED)), RESOURCES),
        ("rest", StandardScaler(), others),
    ]),
    LogisticRegression(max_iter=5000),
)
pca_grid = {"columntransformer__resources__pca__n_components": [1, 2],
            "logisticregression__C": [0.1, 1]}

kmeans_lr = make_pipeline(
    FeatureUnion([
        ("scaled", StandardScaler()),
        ("clusters", make_pipeline(StandardScaler(), KMeans(n_init=10, random_state=SEED))),
    ]),
    StandardScaler(),
    LogisticRegression(max_iter=5000),
)
kmeans_grid = {"featureunion__clusters__kmeans__n_clusters": [2, 4, 8],
               "logisticregression__C": [0.1, 1]}

transform_rows = [ablation14.loc[ablation14["model"].eq("Logistic regression")
                                 & ablation14["features"].eq("All engineered")].iloc[0].to_dict()]
transform_rows[0]["features"] = "All engineered (plain)"
for name, model, grid in [("All engineered + PCA on resources", pca_lr, pca_grid),
                          ("All engineered + K-means distances", kmeans_lr, kmeans_grid)]:
    search, scores = tune(model, grid, train14, all14)
    transform_rows.append(summarize("Logistic regression", name, scores, search.best_params_,
                                    minute=14, n_features=len(all14)))
transforms = pd.DataFrame(transform_rows)
display(transforms[["features"] + METRIC_COLS + ["params"]].round(4))

## 4. L1 feature selection (minute 14)

An L1 penalty drives the coefficients of unhelpful features to exactly zero; a smaller `C` means a stronger penalty and fewer features. We trace CV log loss against the number of kept features, then pick the **smallest model whose log loss is within one standard error of the best** (the one-standard-error rule).

In [ ]:
l1_rows = []
for C in np.logspace(-3, 0, 7):
    model = make_pipeline(StandardScaler(),
                          LogisticRegression(l1_ratio=1, solver="saga", C=C, max_iter=5000))
    scores = cv_scores(model, train14, all14)
    model.fit(train14[all14], train14["hasWon"])
    kept = [c for c, w in zip(all14, model[-1].coef_[0]) if w != 0]
    l1_rows.append({"C": C, "n_kept": len(kept), "kept": kept,
                    "log_loss": scores["log_loss"].mean(),
                    "log_loss_se": scores["log_loss"].std(ddof=1) / np.sqrt(len(scores["log_loss"])),
                    "accuracy": scores["accuracy"].mean()})
l1_path = pd.DataFrame(l1_rows)

best = l1_path.loc[l1_path["log_loss"].idxmin()]
within = l1_path.loc[l1_path["log_loss"] <= best["log_loss"] + best["log_loss_se"]]
chosen = within.loc[within["n_kept"].idxmin()]
selected = chosen["kept"]

display(l1_path[["C", "n_kept", "log_loss", "log_loss_se", "accuracy"]].round(4))
print(f"Chosen C = {chosen['C']:.4f}: {len(selected)} of {len(all14)} features kept")
print(selected)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.errorbar(l1_path["n_kept"], l1_path["log_loss"], yerr=l1_path["log_loss_se"], marker="o", capsize=3)
ax.axhline(best["log_loss"] + best["log_loss_se"], color="gray", linestyle="--", linewidth=1,
           label="best + 1 standard error")
ax.scatter([chosen["n_kept"]], [chosen["log_loss"]], s=120, facecolors="none", edgecolors="#d97757",
           linewidths=2, label="chosen", zorder=3)
ax.set_xlabel("Features kept by L1 logistic regression")
ax.set_ylabel("CV log loss (lower is better)")
ax.set_title("L1 feature selection at minute 14")
ax.legend()
fig.tight_layout()
fig.savefig(figure_dir / "l1_selection_path.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
selection_rows = []
for model_name, model, grid in [("Logistic regression", lr, lr_grid), ("Decision tree", dt, dt_grid)]:
    search, scores = tune(model, grid, train14, selected)
    selection_rows.append(summarize(model_name, "L1-selected", scores, search.best_params_,
                                    minute=14, n_features=len(selected)))

minute14_summary = pd.concat([ablation14, transforms.iloc[1:], pd.DataFrame(selection_rows)],
                             ignore_index=True)
display(minute14_summary[["model", "features", "n_features"] + METRIC_COLS].round(4))
minute14_summary.assign(params=minute14_summary["params"].astype(str)).to_csv(
    report_dir / "fe_minute14_results.csv", index=False)

## 5. Ablation across prediction minutes

Later in the game, more objectives have been taken (inner turrets, inhibitors, Baron) and there is a longer history to summarize, so the new features may matter more. Minute 10 has no trajectory features.

In [ ]:
tables = {14: (table14, groups14), 20: (table20, groups20)}
for minute in [10, 18]:
    tables[minute] = build_features(raw, minute)

by_minute = []
for minute in [10, 14, 18, 20]:
    if minute == 14:
        by_minute.append(ablation14)
        continue
    table, groups = tables[minute]
    data = table.loc[table["gameId"].isin(train_ids)].reset_index(drop=True)
    by_minute.append(run_ablation(data, groups, minute))
by_minute = pd.concat(by_minute, ignore_index=True)
by_minute.assign(params=by_minute["params"].astype(str)).to_csv(
    report_dir / "fe_by_minute.csv", index=False)

display(by_minute.loc[by_minute["model"].ne("Gold-lead rule")]
        .pivot_table(index=["model", "features"], columns="minute", values="log_loss").round(4))
display(by_minute.loc[by_minute["model"].ne("Gold-lead rule")]
        .pivot_table(index=["model", "features"], columns="minute", values="accuracy").round(4))

In [ ]:
base = (by_minute.loc[by_minute["features"].eq("Notebook 01 features")]
        .set_index(["model", "minute"])[["accuracy", "log_loss"]])
gain = by_minute.loc[by_minute["features"].isin(["+ objective", "+ trajectory", "All engineered"])].copy()
gain = gain.join(base, on=["model", "minute"], rsuffix="_base")
gain["accuracy gain (pp)"] = (gain["accuracy"] - gain["accuracy_base"]) * 100
gain["log loss reduction"] = gain["log_loss_base"] - gain["log_loss"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, (model_name, part) in zip(axes, gain.groupby("model")):
    sns.barplot(data=part, x="minute", y="log loss reduction", hue="features",
                palette=["#2a9d8f", "#e9c46a", "#264653"], ax=ax)
    ax.axhline(0, color="gray", linewidth=1)
    ax.set_title(model_name)
    ax.set_ylabel("Log loss reduction vs. notebook 01 features")
fig.suptitle("What the new feature groups add at each prediction minute (higher is better)")
fig.tight_layout()
fig.savefig(figure_dir / "fe_gain_by_minute.png", dpi=200, bbox_inches="tight")
plt.show()

display(gain[["model", "features", "minute", "accuracy gain (pp)", "log loss reduction"]].round(4))

## 6. Save feature tables for the other notebooks

One CSV per minute with every eligible match (training and test), its split, and all features. Features are computed row by row from each match's own history, with nothing fitted, so building them for test matches does not leak information. Models must still be fit on training rows only. The files go to `data/processed/` (not committed), so rerun this notebook to regenerate them. They can also be built directly with `src.features.build_features(raw, minute)`.

In [ ]:
for minute, (table, groups) in sorted(tables.items()):
    out = table.merge(split_ids, on="gameId", validate="one_to_one")
    out.to_csv(processed_dir / f"features_minute{minute}.csv", index=False)
    print(f"minute {minute}: {out.shape[0]:,} matches, {out.shape[1] - 3} features "
          f"({out['split'].value_counts().to_dict()})")

## 7. Findings

FINDINGS_PLACEHOLDER